# CHB-MIT preprocessing • mv-PIKN
**Colab notebook · revision 2.2 · 28 October 2026**

For the completed run, run **only Section 7B** below after mounting Google Drive. Its imports and saved-run loading are self-contained. If needed, install `h5py`, `numpy`, and `pandas` first. Do not rerun preprocessing to obtain this audit.

Section 7B distinguishes annotation-eligible forecast targets, training labels, and signal validity. It preserves all processed data, target eligibility, fold assignments, scalers and original status. Two new reports explain the earlier generic QC warning. They do not automatically approve the evaluation policy.

For a fresh runtime that needs further EDF processing, use Sections 1–7 including the revision 2.1 montage patch in 3B, then run 7B. The frozen processing configuration remains version 2.0 for compatible checkpoint reuse.


## 1 · Install dependencies and mount Drive
Use a CPU runtime; preprocessing does not require a GPU. The notebook records installed package versions with each run.

In [ ]:
%pip -q install "pyedflib>=0.1.40" "h5py>=3.10" "scipy>=1.11" "pandas>=2.0" "wfdb>=4.1" tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

## 2 · Paths and optional download
Your supplied AWS command is incorporated here. Leave `DOWNLOAD_DATA = False` if the sync is complete. `aws s3 sync` can continue an interrupted download.

PhysioNet lists approximately **42.6 GB** of uncompressed source data. Retaining filtered float32 signals can require roughly another **60–65 GB before compression**, depending on the selected cohort. This notebook stores each processed recording once, uses bounded memory, and stages one EDF locally at a time. Results go into a separate output folder.

In [ ]:
from pathlib import Path
import subprocess, sys

RAW_ROOT = Path('/content/gdrive/MyDrive/CHB-MIT')
OUTPUT_ROOT = Path('/content/gdrive/MyDrive/mvPIKN_preprocessing')
DOWNLOAD_DATA = False

if DOWNLOAD_DATA:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'awscli'], check=True)
    subprocess.run(['aws','s3','sync','--no-sign-request',
                    's3://physionet-open/chbmit/1.0.0/', str(RAW_ROOT)], check=True)
print('Raw data:', RAW_ROOT)
print('Outputs:', OUTPUT_ROOT)

## 3 · Pipeline implementation
This cell defines the audit, split proposal, preprocessing, and data loader functions. It does not process recordings until the execution cells below.

**Signal choices:** fixed 18-channel bipolar montage; explicit polarity correction for reversed derivations; duplicate channels checked for equality, with the documented chb12/chb15 exception installed in Section 3B; no interpolation of missing channels. Nonstandard montages remain in the audit with exclusion reasons. CHB-MIT is already 256 Hz: unexpected sampling rates are flagged for review rather than silently resampled.

**Causal filter:** fourth-order Butterworth bandpass design, 0.5–40 Hz, implemented with forward second-order sections. Filter state carries across processing batches and resets at each EDF boundary. The first 30 seconds of each EDF are marked as warm-up. Windows and Koopman pairs never bridge files or gaps. Thirty seconds is a configurable initial transient exclusion, not a validated universal settling time.

**Quality control:** amplitudes above 500 µV and deviations above 6 raw MAD are flags, not automatic rejection. MAD is calculated separately in each completed window. Any channel flat for at least one second, nonfinite samples, and filter warm-up/recovery invalidate a window. All windows remain in the output with masks. No whole-file 35% contamination deletion is applied.

**Features:** Welch PSD with 512-sample Hann segments, 256-sample overlap, constant detrending, density scaling. Four relative bands use [0.5,4), [4,8), [8,13), [13,30) Hz and a 0.5–40 Hz denominator. Their sum can therefore be less than one. The fifth feature is natural-log mean-square amplitude in µV², floored at 1e-12. Features use filtered, unnormalised signals.

In [ ]:
"""CHB-MIT preprocessing v2.0. Shared signal bank and nested subject folds."""
from pathlib import Path
from datetime import datetime, timezone
import re, json, hashlib, shutil, tempfile, platform, importlib.metadata
import numpy as np
import pandas as pd
import scipy
from scipy.signal import butter, sosfilt, sosfilt_zi, welch
import pyedflib
import h5py
from tqdm.auto import tqdm

CHANNELS = ['FP1-F7','F7-T7','T7-P7','P7-O1','FP1-F3','F3-C3',
            'C3-P3','P3-O1','FP2-F4','F4-C4','C4-P4','P4-O2',
            'FP2-F8','F8-T8','T8-P8','P8-O2','FZ-CZ','CZ-PZ']
DEFAULTS = dict(fs=256, window_s=4, bandpass_hz=[0.5,40], filter_order=4,
    warmup_s=30, sph_s=300, sop_s=1800, fusion_history_s=300,
    postictal_s=14400, interictal_before_s=14400, lead_gap_s=14400,
    min_context_fraction=0.95, max_context_gap_s=60,
    min_train_events=2, min_val_events=1, min_test_events=1,
    split_fractions=[0.60,0.20,0.20], amplitude_flag_uv=500,
    mad_flag_multiplier=6, flatline_s=1, flatline_tolerance_uv=0.01,
    batch_windows=64, exclude_cases=['chb21'], channels=CHANNELS,
    pipeline_version='2.0', inner_folds=3, fold_seed=20261028)

def reconcile_annotation_index(path, info, seizure_files):
    """Narrow, evidence-checked correction; unrelated inconsistencies still fail."""
    import wfdb
    if path.parent.name!='chb07' or path.name not in ['chb07_18.edf','chb07_19.edf']:
        raise ValueError('Summary contradicts RECORDS-WITH-SEIZURES')
    expected=[] if path.name=='chb07_18.edf' else [(13688.,13831.)]
    if info['events']!=expected:
        raise ValueError('chb07 summary differs from documented correction')
    if 'chb07/chb07_18.edf' not in seizure_files or 'chb07/chb07_19.edf' in seizure_files:
        raise ValueError('Source index changed; re-review correction')
    if (path.parent/'chb07_18.edf.seizures').exists():
        raise ValueError('Unexpected chb07_18 annotation; re-review correction')
    ann=wfdb.rdann(str(path.parent/'chb07_19.edf'),'seizures')
    if ann.fs!=256: raise ValueError('Unexpected annotation sample rate')
    starts=[float(t/ann.fs) for t,s in zip(ann.sample,ann.symbol) if s=='[']
    ends=[float(t/ann.fs) for t,s in zip(ann.sample,ann.symbol) if s==']']
    if starts!=[13688.] or ends!=[13831.]:
        raise ValueError('chb07 binary annotation contradicts correction')
    return dict(file=str(path.parent.name+'/'+path.name),
        resolution='Use patient summary corroborated by chb07_19.edf.seizures; global index mismatch',
        annotation_sha256=sha256_file(path.parent/'chb07_19.edf.seizures'),
        summary_url='https://physionet.org/files/chbmit/1.0.0/chb07/chb07-summary.txt',
        index_url='https://physionet.org/files/chbmit/1.0.0/RECORDS-WITH-SEIZURES')

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, default=str))

def canonical(label):
    label = re.sub(r'^EEG\s*', '', label.strip().upper())
    label = re.sub(r'\s+', '', label)
    aliases = {'T3':'T7','T4':'T8','T5':'P7','T6':'P8'}
    return '-'.join(aliases.get(x,x) for x in label.split('-'))

def unit_factor(unit):
    unit = unit.strip().lower().replace('µ','u').replace('μ','u')
    return {'uv':1., 'mv':1000., 'v':1e6}.get(unit)

def mapping(labels, channels):
    normalized = [canonical(x) for x in labels]
    result = []
    for name in channels:
        direct = [i for i,x in enumerate(normalized) if x == name]
        rev = '-'.join(name.split('-')[::-1])
        reverse = [i for i,x in enumerate(normalized) if x == rev]
        matches, sign = (direct,1) if direct else (reverse,-1)
        result.append(dict(name=name, indices=matches, sign=sign))
    return result

def parse_summary(text):
    result = {}
    blocks = re.split(r'File Name:\s*', text)[1:]
    for block in blocks:
        name = block.splitlines()[0].strip()
        count = re.search(r'Number of Seizures in File:\s*(\d+)',block)
        starts = re.findall(r'Seizure(?:\s+\d+)? Start Time:\s*([\d.]+)\s*seconds',block)
        ends = re.findall(r'Seizure(?:\s+\d+)? End Time:\s*([\d.]+)\s*seconds',block)
        if not count or len(starts)!=int(count[1]) or len(ends)!=int(count[1]):
            raise ValueError(f'Annotation count mismatch in {name}')
        if name in result:
            raise ValueError(f'Duplicate summary entry: {name}')
        clock = re.search(r'File Start Time:\s*(\d+):(\d+):(\d+)',block)
        result[name] = dict(events=list(zip(map(float,starts),map(float,ends))),
            clock_s=(int(clock[1])*3600+int(clock[2])*60+int(clock[3]))%86400 if clock else None)
    return result

def covered(intervals, start, end):
    """Union coverage and largest gap; never bridge missing recordings."""
    cursor, total, gaps = start, 0., []
    for a,b in sorted(intervals):
        a,b = max(a,start), min(b,end)
        if b<=a: continue
        if a>cursor: gaps.append(a-cursor)
        total += max(0.,b-max(a,cursor))
        cursor=max(cursor,b)
    if cursor<end: gaps.append(end-cursor)
    return total/(end-start), max(gaps,default=0.)

def audit_dataset(root, output_root, cfg):
    root, output_root = Path(root), Path(output_root)
    if not root.is_dir(): raise FileNotFoundError(root)
    expected_path, seizure_path = root/'RECORDS', root/'RECORDS-WITH-SEIZURES'
    if not expected_path.exists() or not seizure_path.exists():
        raise RuntimeError('Finish the sync: RECORDS and RECORDS-WITH-SEIZURES are required.')
    expected = {x.strip() for x in expected_path.read_text().splitlines() if x.strip().endswith('.edf')}
    seizure_files = {x.strip() for x in seizure_path.read_text().splitlines() if x.strip()}
    actual = {str(p.relative_to(root)) for p in root.glob('chb*/*.edf')}
    if not actual: raise RuntimeError('No EDF recordings found under chbXX directories.')
    run = output_root / ('run_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    run.mkdir(parents=True)
    write_json(run/'config.json', cfg)
    versions = {p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','pyedflib','h5py']}
    write_json(run/'environment.json', dict(python=platform.python_version(),packages=versions))
    shutil.copy2(expected_path,run/'source_RECORDS.txt')
    shutil.copy2(seizure_path,run/'source_RECORDS_WITH_SEIZURES.txt')
    rows, events, problems, corrections = [], [], [], []
    for rel in sorted(expected-actual):
        if rel.split('/')[0] not in cfg['exclude_cases']:
            problems.append(dict(file=rel,problem='missing_download',blocking=True))
    for rel in sorted(actual-expected):
        problems.append(dict(file=rel,problem='not_in_RECORDS',blocking=True))
    for case_dir in sorted(root.glob('chb[0-9][0-9]')):
        if not case_dir.is_dir(): continue
        case = case_dir.name
        summary_file = case_dir/(case+'-summary.txt')
        try:
            summary = parse_summary(summary_file.read_text(errors='strict'))
            shutil.copy2(summary_file,run/summary_file.name)
        except Exception as exc:
            summary = {}
            problems.append(dict(file=case,problem='summary_error: '+str(exc),blocking=case not in cfg['exclude_cases']))
        for path in sorted(case_dir.glob('*.edf')):
            rel = str(path.relative_to(root)); excluded = case in cfg['exclude_cases']
            try:
                with pyedflib.EdfReader(str(path)) as reader:
                    labels = reader.getSignalLabels()
                    maps = mapping(labels,cfg['channels'])
                    start = reader.getStartdatetime()
                    duration = float(reader.file_duration)
                    frequencies = reader.getSampleFrequencies().tolist()
                    units = [reader.getPhysicalDimension(i) for i in range(reader.signals_in_file)]
                    counts = reader.getNSamples().tolist()
                info = summary.get(path.name)
                # chb24 summary lists seizure files only. The official global list
                # supplies a documented negative annotation for other files.
                if info is None:
                    if rel in seizure_files or rel not in expected:
                        raise ValueError('Missing seizure annotation')
                    info = dict(events=[],clock_s=None)
                    annotation_source = 'RECORDS-WITH-SEIZURES negative'
                else: annotation_source = summary_file.name
                if bool(info['events']) != (rel in seizure_files):
                    corrections.append(reconcile_annotation_index(path,info,seizure_files))
                    annotation_source += ' + verified binary annotation correction'
                if info['clock_s'] is not None:
                    sec = start.hour*3600+start.minute*60+start.second
                    if min(abs(sec-info['clock_s']),86400-abs(sec-info['clock_s']))>1:
                        raise ValueError('EDF and summary start clocks disagree')
                for a,b in info['events']:
                    if not 0<=a<b<=duration: raise ValueError('Seizure outside EDF duration')
                missing = [x['name'] for x in maps if not x['indices']]
                used = [i for m in maps for i in m['indices']]
                signal_reason = []
                if missing: signal_reason.append('missing_channels:'+','.join(missing))
                if any(abs(frequencies[i]-cfg['fs'])>1e-8 for i in used): signal_reason.append('unexpected_sampling_rate')
                if any(unit_factor(units[i]) is None for i in used): signal_reason.append('unknown_units')
                if any(abs(counts[i]-duration*cfg['fs'])>1 for i in used): signal_reason.append('sample_count_mismatch')
                row = dict(case=case,subject='chb01' if case=='chb21' else case,
                    relative_path=rel,bytes=path.stat().st_size,mtime_ns=path.stat().st_mtime_ns,
                    start_iso=start.isoformat(),epoch_s=(start-datetime(1970,1,1)).total_seconds(),
                    duration_s=duration,labels=labels,units=units,mapping=maps,
                    annotation_source=annotation_source,seizures=len(info['events']),
                    excluded_case=excluded,signal_ok=not signal_reason,
                    signal_reason=';'.join(signal_reason),split='unassigned')
                rows.append(row)
                for n,(a,b) in enumerate(info['events']):
                    events.append(dict(case=case,relative_path=rel,event_id=f'{path.stem}_sz{n+1:02d}',
                        onset_epoch=row['epoch_s']+a,offset_epoch=row['epoch_s']+b))
            except Exception as exc:
                problems.append(dict(file=rel,problem=type(exc).__name__+': '+str(exc),blocking=not excluded))
    # Use EDF dates; PhysioNet preserves within-case relative time.
    for case in sorted({r['case'] for r in rows}):
        rr = sorted([r for r in rows if r['case']==case],key=lambda r:r['epoch_s'])
        origin = rr[0]['epoch_s']; previous_end = None
        for r in rr:
            r['start_s']=r['epoch_s']-origin; r['end_s']=r['start_s']+r['duration_s']
            r['gap_before_s']=None if previous_end is None else r['start_s']-previous_end
            if previous_end is not None and r['start_s']<previous_end-1/cfg['fs']:
                problems.append(dict(file=r['relative_path'],problem='overlapping_EDF_times',blocking=not r['excluded_case']))
            previous_end=max(previous_end or r['end_s'],r['end_s'])
        ee = sorted([e for e in events if e['case']==case],key=lambda e:e['onset_epoch'])
        coverage = [(r['start_s']+cfg['warmup_s'],r['end_s']) for r in rr if r['signal_ok']]
        last_offset = None
        for e in ee:
            e['onset_s']=e.pop('onset_epoch')-origin; e['offset_s']=e.pop('offset_epoch')-origin
            # Leading event means separation from the previous ANNOTATED seizure;
            # lack of observations is reported separately, never assumed seizure-free.
            e['lead_by_annotations']=last_offset is None or e['onset_s']-last_offset>=cfg['lead_gap_s']
            a=e['onset_s']-cfg['sph_s']-cfg['sop_s']-cfg['fusion_history_s']-cfg['window_s']
            b=e['onset_s']-cfg['sph_s']
            frac,gap=covered(coverage,a,b)
            e['context_fraction']=frac; e['max_context_gap_s']=gap
            e['history_fraction_4h']=covered(coverage,e['onset_s']-cfg['lead_gap_s'],e['onset_s'])[0]
            e['eligible_target']=bool(e['lead_by_annotations'] and frac>=cfg['min_context_fraction'] and gap<=cfg['max_context_gap_s'])
            e['split']='unassigned'
            last_offset=max(last_offset or e['offset_s'],e['offset_s'])
    write_json(run/'inventory.json',rows); write_json(run/'events.json',events)
    write_json(run/'annotation_corrections.json',corrections)
    pd.DataFrame(rows).drop(columns=['mapping'],errors='ignore').to_csv(run/'inventory.csv',index=False)
    pd.DataFrame(events).to_csv(run/'events.csv',index=False)
    pd.DataFrame(problems,columns=['file','problem','blocking']).to_csv(run/'audit_issues.csv',index=False)
    coverage_rows=[]
    for case in sorted({r['case'] for r in rows}):
        rr=[r for r in rows if r['case']==case]; ee=[e for e in events if e['case']==case]
        coverage_rows.append(dict(case=case,files=len(rr),recorded_hours=sum(r['duration_s'] for r in rr)/3600,
            montage_usable_files=sum(r['signal_ok'] for r in rr),annotated_seizures=len(ee),
            eligible_targets=sum(e['eligible_target'] for e in ee),excluded_case=rr[0]['excluded_case']))
    pd.DataFrame(coverage_rows).to_csv(run/'cohort_audit.csv',index=False)
    return run,rows,events,problems

def propose_splits(rows,events,cfg):
    """Chronological whole-file boundaries; keep forecast contexts and clusters together.

    Candidate search uses timestamps/annotations only, never signal features or model scores.
    Targets from a <=4h seizure cluster stay in one partition. A valid split requires
    >=2/1/1 eligible events in train/validation/test. No forced fallback split.
    """
    report=[]
    for case in sorted({r['case'] for r in rows}):
        rr=sorted([r for r in rows if r['case']==case],key=lambda r:r['start_s'])
        ee=sorted([e for e in events if e['case']==case],key=lambda e:e['onset_s'])
        if rr[0]['excluded_case']:
            report.append(dict(case=case,status='excluded_case',reason='repeated subject session policy'));continue
        # Merge protected forecast contexts and close seizure episodes.
        protected=[]
        pre=cfg['sph_s']+cfg['sop_s']+cfg['fusion_history_s']+cfg['window_s']+cfg['warmup_s']
        for e in ee:
            a,b=e['onset_s']-pre,e['offset_s']
            if protected and e['onset_s']-protected[-1][1]<cfg['lead_gap_s']:
                protected[-1][1]=max(b,protected[-1][1])
            else: protected.append([a,b])
        candidates=[]
        for k in range(1,len(rr)):
            boundary=(rr[k-1]['end_s']+rr[k]['start_s'])/2
            if not any(a<boundary<b for a,b in protected): candidates.append((k,boundary))
        best=None; total=sum(r['duration_s'] for r in rr)
        eligible=[e for e in ee if e['eligible_target']]
        for i,b1 in candidates:
            for j,b2 in candidates:
                if j<=i: continue
                counts=[sum(e['onset_s']<b1 for e in eligible),
                        sum(b1<=e['onset_s']<b2 for e in eligible),
                        sum(e['onset_s']>=b2 for e in eligible)]
                if any(n<m for n,m in zip(counts,[cfg['min_train_events'],cfg['min_val_events'],cfg['min_test_events']])): continue
                durations=[sum(r['duration_s'] for r in rr[:i]),sum(r['duration_s'] for r in rr[i:j]),sum(r['duration_s'] for r in rr[j:])]
                score=sum((x/total-y)**2 for x,y in zip(durations,cfg['split_fractions']))
                if best is None or score<best[0]: best=(score,i,j,b1,b2,counts,durations)
        if best is None:
            report.append(dict(case=case,status='not_feasible',reason='insufficient separated events or safe whole-file boundaries'));continue
        _,i,j,b1,b2,counts,durations=best
        for k,r in enumerate(rr): r['split']='train' if k<i else 'validation' if k<j else 'test'
        for e in ee: e['split']='train' if e['onset_s']<b1 else 'validation' if e['onset_s']<b2 else 'test'
        report.append(dict(case=case,status='proposed',reason='',train_events=counts[0],validation_events=counts[1],test_events=counts[2],
            train_hours=durations[0]/3600,validation_hours=durations[1]/3600,test_hours=durations[2]/3600))
    return pd.DataFrame(report)

def assign_labels(starts,ends,events,split,cfg):
    """Labels: -1 excluded/uncertain; 0 distant interictal; 1 preictal; 2 ictal.
    Decisions occur at window END. Partial ictal or preictal boundary windows
    are excluded from supervised learning, but retained for timeline replay.
    """
    y=np.full(len(starts),-1,np.int8)
    distant=np.ones(len(starts),bool); ictal=np.zeros(len(starts),bool)
    intersects_ictal=np.zeros(len(starts),bool); post=np.zeros(len(starts),bool)
    pre=np.zeros(len(starts),bool); target=np.full(len(starts),'',dtype=object)
    for e in events:
        a,b=e['onset_s'],e['offset_s']
        intersects_ictal |= (starts<b)&(ends>a)
        ictal |= (starts>=a)&(ends<=b)
        post |= (starts<b+cfg['postictal_s'])&(ends>b)
        distant &= (ends<=a-cfg['interictal_before_s'])|(starts>=b+cfg['postictal_s'])
        if e['eligible_target'] and e['split']==split:
            m=(starts>=a-cfg['sph_s']-cfg['sop_s'])&(ends<=a-cfg['sph_s'])
            pre |= m; target[m]=e['event_id']
    y[distant]=0
    y[pre & ~post & ~intersects_ictal]=1
    y[ictal]=2
    target[y!=1]=''
    return y,target

def flatline_mask(windows,cfg):
    out=np.zeros(len(windows),bool)
    need=int(cfg['flatline_s']*cfg['fs'])-1
    for i,w in enumerate(windows):
        for channel in w:
            same=np.abs(np.diff(channel))<=cfg['flatline_tolerance_uv']
            edges=np.diff(np.r_[False,same,False].astype(np.int8))
            if np.any(np.flatnonzero(edges==-1)-np.flatnonzero(edges==1)>=need):
                out[i]=True;break
    return out

def spectral_features(windows,fs):
    freq,psd=welch(windows,fs=fs,window='hann',nperseg=512,noverlap=256,
        detrend='constant',scaling='density',axis=-1)
    df=freq[1]-freq[0]
    full=(freq>=0.5)&(freq<=40)
    denominator=psd[...,full].sum(axis=-1)*df
    bp=[]
    for lo,hi in [(0.5,4),(4,8),(8,13),(13,30)]:
        mask=(freq>=lo)&(freq<hi)
        bp.append(psd[...,mask].sum(axis=-1)*df/np.maximum(denominator,1e-12))
    logpower=np.log(np.maximum(np.mean(windows.astype(np.float64)**2,axis=-1),1e-12))
    return np.stack(bp+[logpower],axis=-1).astype(np.float32)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

def process_file(root,row,events,cfg,dest,recording_intervals=None):
    """Bounded-memory filtering. Reset only at EDF start, preserving state across batches."""
    path=Path(root)/row['relative_path']
    if path.stat().st_size!=row['bytes'] or path.stat().st_mtime_ns!=row['mtime_ns']:
        raise RuntimeError('Source changed since audit: '+row['relative_path'])
    if recording_intervals is None: recording_intervals=[(row['start_s'],row['end_s'])]
    fs=cfg['fs']; length=int(cfg['window_s']*fs); C=len(cfg['channels'])
    N=int(row['duration_s']*fs)//length
    if N==0: raise ValueError('No full windows')
    sos=butter(cfg['filter_order'],cfg['bandpass_hz'],btype='bandpass',fs=fs,output='sos')
    zi=None; previous_valid=False; previous_y=-1
    stats=dict(n_samples=0,raw_sum=np.zeros(C),raw_sq=np.zeros(C),n_features=0,
        feature_sum=np.zeros((C,5)),feature_sq=np.zeros((C,5)))
    with tempfile.TemporaryDirectory(prefix='chbmit_') as tmp:
        local=Path(tmp)/path.name
        shutil.copy2(path,local)  # Avoid many random reads through the Drive mount.
        digest=sha256_file(local)
        out=Path(tmp)/'data.h5'
        with pyedflib.EdfReader(str(local)) as reader, h5py.File(out,'w') as h:
            h.attrs['source_sha256']=digest; h.attrs['relative_path']=row['relative_path']
            h.attrs['case']=row['case']; h.attrs['split']=row['split']; h.attrs['fs']=fs
            h.attrs['channels_json']=json.dumps(cfg['channels']);h.attrs['raw_units']='microvolts'
            h.attrs['feature_names_json']=json.dumps(['relative_delta','relative_theta','relative_alpha','relative_beta','ln_mean_square_uv2'])
            h.attrs['config_json']=json.dumps(cfg,sort_keys=True)
            h.create_dataset('raw_uv',(N,C,length),dtype='f4',chunks=(min(16,N),C,length),compression='gzip',compression_opts=1,shuffle=True)
            h.create_dataset('features',(N,C,5),dtype='f4')
            for name,kind in [('label','i1'),('valid_signal','?'),('amplitude_flag','?'),('mad_flag','?'),('flatline_flag','?'),('warmup_flag','?'),('nonfinite_flag','?'),('pair_with_previous','?'),('supervised_pair_with_previous','?')]:
                h.create_dataset(name,(N,),dtype=kind)
            h.create_dataset('window_start_s',(N,),dtype='f8');h.create_dataset('decision_s',(N,),dtype='f8')
            h.create_dataset('future_annotation_coverage',(N,),dtype='f4')
            h.create_dataset('future_annotation_max_gap_s',(N,),dtype='f4')
            h.create_dataset('target_event_id',(N,),dtype=h5py.string_dtype('utf-8'))
            counts={-1:0,0:0,1:0,2:0}; valid_count=0; flags_count=dict(amplitude=0,mad=0,flatline=0,warmup=0,nonfinite=0)
            contamination_until=-1
            for first in range(0,N,cfg['batch_windows']):
                last=min(N,first+cfg['batch_windows']); n=(last-first)*length; start_sample=first*length
                traces=[]
                for m in row['mapping']:
                    candidates=[reader.readSignal(i,start_sample,n)*unit_factor(row['units'][i])*m['sign'] for i in m['indices']]
                    # Duplicate channel names must actually encode the same signal.
                    if any(not np.allclose(candidates[0],x,rtol=0,atol=1e-3,equal_nan=True) for x in candidates[1:]):
                        raise ValueError('Nonidentical duplicate channel: '+m['name'])
                    traces.append(candidates[0])
                x=np.stack(traces)
                if x.shape!=(C,n): raise ValueError('Unexpected signal length')
                native=x.reshape(C,last-first,length).transpose(1,0,2)
                finite=np.isfinite(native).all(axis=(1,2))
                amplitude=(np.abs(native)>cfg['amplitude_flag_uv']).any(axis=(1,2))
                flat=flatline_mask(native,cfg)
                med=np.nanmedian(native,axis=-1,keepdims=True)
                mad=np.nanmedian(np.abs(native-med),axis=-1,keepdims=True)
                # Raw (unscaled) MAD, computed from the current completed window.
                madflag=(np.abs(native-med)>cfg['mad_flag_multiplier']*np.maximum(mad,1e-6)).any(axis=(1,2))
                clean=np.nan_to_num(x,nan=0.,posinf=0.,neginf=0.)
                if zi is None: zi=sosfilt_zi(sos)[:,None,:]*clean[None,:,0,None]
                filtered,zi=sosfilt(sos,clean,axis=-1,zi=zi)
                raw=filtered.reshape(C,last-first,length).transpose(1,0,2).astype(np.float32)
                features=spectral_features(raw,fs)
                local_start=np.arange(first,last)*cfg['window_s']
                starts=row['start_s']+local_start; ends=starts+cfg['window_s']
                future=np.asarray([covered(recording_intervals,t+cfg['sph_s'],t+cfg['sph_s']+cfg['sop_s']) for t in ends])
                warm=local_start<cfg['warmup_s']
                # Exclude recovery after substituted nonfinite samples, using past only.
                recovery=np.zeros(last-first,bool)
                for k,ok in enumerate(finite):
                    if not ok: contamination_until=local_start[k]+cfg['window_s']+cfg['warmup_s']
                    recovery[k]=local_start[k]<contamination_until
                valid=finite & ~flat & ~warm & ~recovery
                y,targets=assign_labels(starts,ends,events,row['split'],cfg)
                pair=valid & np.r_[previous_valid,valid[:-1]]
                supervised=pair & (y>=0) & np.r_[previous_y>=0,y[:-1]>=0]
                previous_valid=bool(valid[-1]);previous_y=int(y[-1])
                sl=slice(first,last)
                for name,data in dict(raw_uv=raw,features=features,label=y,valid_signal=valid,
                    amplitude_flag=amplitude,mad_flag=madflag,flatline_flag=flat,warmup_flag=warm,
                    nonfinite_flag=~finite|recovery,pair_with_previous=pair,
                    supervised_pair_with_previous=supervised,window_start_s=starts,decision_s=ends,target_event_id=targets,
                    future_annotation_coverage=future[:,0],future_annotation_max_gap_s=future[:,1]).items(): h[name][sl]=data
                for cls in counts: counts[cls]+=int(((y==cls)&valid).sum())
                valid_count+=int(valid.sum())
                for key,values in dict(amplitude=amplitude,mad=madflag,flatline=flat,warmup=warm,nonfinite=~finite|recovery).items(): flags_count[key]+=int(values.sum())
                if row['split'] in ['train','pool']:
                    keep=valid & (y>=0)
                    a=raw[keep].astype(np.float64); f=features[keep].astype(np.float64)
                    stats['n_samples']+=a.shape[0]*length
                    stats['raw_sum']+=a.sum(axis=(0,2));stats['raw_sq']+=(a*a).sum(axis=(0,2))
                    stats['n_features']+=f.shape[0]
                    stats['feature_sum']+=f.sum(axis=0);stats['feature_sq']+=(f*f).sum(axis=0)
        dest=Path(dest);dest.parent.mkdir(parents=True,exist_ok=True)
        staging=dest.with_suffix('.h5.partial');shutil.copy2(out,staging);staging.replace(dest)
    summary=dict(case=row['case'],file=row['relative_path'],split=row['split'],source_sha256=digest,
        windows=N,valid_windows=valid_count,valid_hours=valid_count*cfg['window_s']/3600,
        tail_seconds=row['duration_s']-N*cfg['window_s'],**{f'label_{k}_valid':v for k,v in counts.items()},
        **{f'{k}_flag_windows':v for k,v in flags_count.items()})
    return summary,stats

def preprocess(root,run,rows,events,problems,cfg,folds=None):
    run=Path(run)
    if any(p['blocking'] for p in problems):
        raise RuntimeError('Resolve blocking audit_issues.csv entries before preprocessing.')
    pooled=folds is not None
    selected=[r for r in rows if r['split'] in ['train','validation','test','pool'] and r['signal_ok']]
    if not selected: raise RuntimeError('No feasible primary cohort. Review split_report.csv; do not force random window splits.')
    if json.loads((run/'config.json').read_text())!=cfg:
        raise RuntimeError('Configuration changed since audit. Start a new audit/run.')
    bank=run/'processed'
    if pooled and bank.exists() and json.loads((run/'locked_folds.json').read_text())!=folds:
        raise RuntimeError('Fold definitions changed. Start a new run.')
    if bank.exists() and json.loads((run/'locked_inventory.json').read_text())!=rows:
        raise RuntimeError('Split inventory changed. Start a fresh run.')
    bank.mkdir(exist_ok=True)
    write_json(run/'locked_inventory.json',rows);write_json(run/'locked_events.json',events)
    if pooled: write_json(run/'locked_folds.json',folds)
    summaries=[];totals={};failures=[]
    for row in tqdm(selected,desc='Processing EDF recordings'):
        case=row['case'];dest=bank/case/row['split']/(Path(row['relative_path']).stem+'.h5')
        try:
            source=Path(root)/row['relative_path']
            if source.stat().st_size!=row['bytes'] or source.stat().st_mtime_ns!=row['mtime_ns']:
                raise RuntimeError('Source changed since audit')
            checkpoint=dest.with_suffix('.done.json')
            if dest.exists() and checkpoint.exists():
                saved=json.loads(checkpoint.read_text());summary=saved['summary'];stats=saved['stats']
                stats={k:np.asarray(v) if isinstance(v,list) else v for k,v in stats.items()}
                with h5py.File(dest) as h:
                    if h.attrs['source_sha256']!=summary['source_sha256'] or len(h['label'])!=summary['windows']:
                        raise RuntimeError('Checkpoint verification failed')
            else:
                intervals=[(r['start_s'],r['end_s']) for r in rows if r['case']==case and r['split']==row['split']]
                summary,stats=process_file(root,row,[e for e in events if e['case']==case],cfg,dest,intervals)
                temporary=checkpoint.with_suffix('.partial')
                write_json(temporary,dict(summary=summary,stats={k:v.tolist() if isinstance(v,np.ndarray) else v for k,v in stats.items()}))
                temporary.replace(checkpoint)
            summaries.append(summary)
            if case not in totals: totals[case]=stats
            else:
                for key,value in stats.items(): totals[case][key]+=value
        except Exception as exc:
            failures.append(dict(file=row['relative_path'],error=type(exc).__name__+': '+str(exc)))
        pd.DataFrame(summaries).to_csv(run/'processing_report.csv',index=False)
        pd.DataFrame(failures,columns=['file','error']).to_csv(run/'processing_errors.csv',index=False)
    # Do not declare a usable cohort if signal reading or duplicate verification failed.
    if failures:
        write_json(run/'status.json',dict(status='incomplete',errors=len(failures)))
        raise RuntimeError('Processing errors recorded. Retry this run for transient I/O failures; use a fresh audit/run if data or policy changes.')
    for case,s in totals.items():
        if pooled:
            np.savez(bank/case/'sufficient_statistics.npz',**s)
            continue
        if not s['n_samples'] or not s['n_features']: raise RuntimeError(f'No valid labelled training data for {case}')
        rm=s['raw_sum']/s['n_samples'];fm=s['feature_sum']/s['n_features']
        rs=np.sqrt(np.maximum(s['raw_sq']/s['n_samples']-rm*rm,1e-12))
        fs=np.sqrt(np.maximum(s['feature_sq']/s['n_features']-fm*fm,1e-12))
        np.savez(bank/case/'train_scaler.npz',raw_mean=rm,raw_std=rs,feature_mean=fm,feature_std=fs,
                 n_training_samples=s['n_samples'],n_training_windows=s['n_features'])
    # Final event-level eligibility additionally requires valid prediction windows.
    event_report=[]
    for e in events:
        if not e['eligible_target'] or e['split']=='unassigned': continue
        count=0
        for file in (bank/e['case']/e['split']).glob('*.h5'):
            with h5py.File(file,'r') as h:
                ids=h['target_event_id'].asstr()[:]
                count+=int(((ids==e['event_id'])&h['valid_signal'][:]).sum())
        event_report.append(dict(**e,valid_preictal_windows=count,has_valid_preictal_input=count>0))
    pd.DataFrame(event_report).to_csv(run/'event_input_coverage.csv',index=False)
    inadequate=[]
    for case in ([] if pooled else totals):
        for split,minimum in [('train',cfg['min_train_events']),('validation',cfg['min_val_events']),('test',cfg['min_test_events'])]:
            n=sum(x['case']==case and x['split']==split and x['has_valid_preictal_input'] for x in event_report)
            if n<minimum: inadequate.append(dict(case=case,split=split,usable_events=n,required=minimum))
    if pooled:
        inadequate=save_fold_scalers(run,rows,folds,totals,summaries)
        # Keep originally eligible targets visible even if QC removes all their input.
        inadequate += [dict(event_id=e['event_id'],reason='no_valid_preictal_input_after_QC')
                       for e in event_report if not e['has_valid_preictal_input']]
    write_json(run/'status.json',dict(status='review_required' if inadequate else 'preprocessing_complete',
        cohort_issues=inadequate,note='Annotation-based target eligibility; missing-time and quality masks must be retained in event scoring.'))
    print('Saved:',run)
    if inadequate: print('Some splits have insufficient events after QC. Review status.json before training.')
    return pd.DataFrame(summaries)

def load_window(path,index,scaler_path):
    """Apply the supplied TRAINING scaler lazily; never refit on validation/test."""
    with h5py.File(path,'r') as h, np.load(scaler_path) as s:
        if not h['valid_signal'][index]: raise ValueError('Window is marked invalid')
        raw=(h['raw_uv'][index]-s['raw_mean'][:,None])/s['raw_std'][:,None]
        features=(h['features'][index]-s['feature_mean'])/s['feature_std']
        return raw.astype(np.float32),features.astype(np.float32),int(h['label'][index])

def prepare_subject_folds(rows,events,cfg):
    """Targets stay outcome annotations; fold roles live ONLY in separate manifests.

    Every usable subject gets an outer test fold, including zero-target subjects
    for false-alarm/coverage reporting. Inner folds partition development subjects;
    labels never select train/validation samples within an individual's recordings.
    """
    subjects=sorted({r['subject'] for r in rows if not r['excluded_case'] and r['signal_ok']})
    case_subject={r['case']:r['subject'] for r in rows}
    case_excluded={r['case']:r['excluded_case'] for r in rows}
    counts={s:sum(e['eligible_target'] and not case_excluded[e['case']] and case_subject[e['case']]==s for e in events) for s in subjects}
    if len(subjects)<cfg['inner_folds']+1:
        raise ValueError('Insufficient subjects for configured nested folds')
    for r in rows:
        r['split']='pool' if not r['excluded_case'] and r['subject'] in subjects else 'unassigned'
    for e in events:
        e['subject']=case_subject[e['case']]
        e['split']='pool' if e['subject'] in subjects and not case_excluded[e['case']] else 'unassigned'
    folds=[]
    for heldout in subjects:
        development=[s for s in subjects if s!=heldout]
        # Randomise tie order reproducibly, then balance development event counts.
        # The held-out subject's event count does not affect inner fold construction.
        rng=np.random.default_rng(cfg['fold_seed'])
        order=list(rng.permutation(development));order.sort(key=lambda s:-counts[s])
        groups=[[] for _ in range(cfg['inner_folds'])];loads=[0]*len(groups)
        for s in order:
            k=min(range(len(groups)),key=lambda j:(loads[j],len(groups[j]),j))
            groups[k].append(str(s));loads[k]+=counts[s]
        inner=[]
        for j,validation in enumerate(groups):
            train=[s for s in development if s not in validation]
            if not train or not validation or sum(counts[s] for s in train)<2 or sum(counts[s] for s in validation)<1:
                raise ValueError(f'Insufficient independent development targets in {heldout}, inner fold {j}; review protocol')
            inner.append(dict(id=f'inner_{j+1:02d}',train_subjects=train,validation_subjects=sorted(validation)))
        folds.append(dict(id='outer_'+heldout,test_subjects=[heldout],development_subjects=development,
            test_eligible_targets=counts[heldout],test_has_targets=counts[heldout]>0,inner_folds=inner))
    return folds

def fold_table(folds):
    return pd.DataFrame([dict(fold=f['id'],test_subject=f['test_subjects'][0],
        test_eligible_targets=f['test_eligible_targets'],development_subjects=len(f['development_subjects']),
        inner_folds=len(f['inner_folds']),test_role='event_and_alarm_scoring' if f['test_has_targets'] else 'alarm_and_coverage_only') for f in folds])

def save_fold_scalers(run,rows,folds,totals,summaries):
    """Pool sufficient statistics ONLY from each fit's named training subjects."""
    run=Path(run);case_subject={r['case']:r['subject'] for r in rows};issues=[]
    for fold in folds:
        scopes=[('outer_refit',fold['development_subjects'])]
        scopes += [(inner['id'],inner['train_subjects']) for inner in fold['inner_folds']]
        for scope,training in scopes:
            fit_cases=[c for c in totals if case_subject[c] in training]
            cls0=sum(s['label_0_valid'] for s in summaries if s['case'] in fit_cases)
            cls1=sum(s['label_1_valid'] for s in summaries if s['case'] in fit_cases)
            if not fit_cases or not cls0 or not cls1:
                issues.append(dict(fold=fold['id'],scope=scope,reason='missing_valid_training_class'));continue
            s={k:sum((totals[c][k] for c in fit_cases)) for k in totals[fit_cases[0]]}
            if not s['n_samples'] or not s['n_features']:
                issues.append(dict(fold=fold['id'],scope=scope,reason='empty_training_statistics'));continue
            rm=s['raw_sum']/s['n_samples'];fm=s['feature_sum']/s['n_features']
            rs=np.sqrt(np.maximum(s['raw_sq']/s['n_samples']-rm*rm,1e-12))
            fs=np.sqrt(np.maximum(s['feature_sq']/s['n_features']-fm*fm,1e-12))
            destination=run/'folds'/fold['id']/scope;destination.mkdir(parents=True,exist_ok=True)
            np.savez(destination/'train_scaler.npz',raw_mean=rm,raw_std=rs,feature_mean=fm,feature_std=fs,
                n_training_samples=s['n_samples'],n_training_windows=s['n_features'],
                training_subjects=np.asarray(training),test_subjects=np.asarray(fold['test_subjects']))
            write_json(destination/'fit_provenance.json',dict(training_subjects=training,training_cases=fit_cases,
                class_0_windows=cls0,class_1_windows=cls1,weighting='pooled valid labelled training samples'))
    return issues

def fold_files(run,inventory,fold,scope='outer_refit',role='train'):
    """Explicit file lists; never use a global random-window DataLoader split."""
    if scope=='outer_refit':
        allowed={'train':fold['development_subjects'],'test':fold['test_subjects']}
    else:
        inner=next(x for x in fold['inner_folds'] if x['id']==scope)
        allowed={'train':inner['train_subjects'],'validation':inner['validation_subjects']}
    if role not in allowed: raise ValueError('Role not allowed for this fit scope')
    return sorted(str(Path(run)/'processed'/r['case']/'pool'/(Path(r['relative_path']).stem+'.h5'))
                  for r in inventory if r['subject'] in allowed[role] and not r['excluded_case'] and r['signal_ok'])


## 3B · Install the resume-compatible montage selection amendment
All 50 reported failures concern T8-P8: 11 recordings in chb12 and 39 in chb15. The public summaries identify two occurrences, and sampled recordings confirm they can carry different signals. Equality is therefore not a valid prerequisite for choosing one montage entry.

For these two cases only, select the unique entry **immediately between F8-T8 and P8-O2**. Omit the supplemental occurrence immediately after FT10-T8. Both label neighbourhoods must match, and the selected entry must be the original processor's first candidate. Unknown layouts or other conflicting duplicate channels still fail.

This is an explicit montage convention, not a claim that the omitted trace is corrupt or that both traces are equivalent. No averaging, swapping based on signal quality, or model-performance selection is performed. Existing successful checkpoints used the same retained entry; their data and statistics are compatible. Reprocessed files and the run-level JSON document the amendment.

Run this cell after Section 3's definitions every time the runtime restarts. Then resume Sections 4–7. A new audit is not required for this specific amendment.

Sources: [chb12 montage summary](https://physionet.org/files/chbmit/1.0.0/chb12/chb12-summary.txt), [chb15 montage summary](https://physionet.org/files/chbmit/1.0.0/chb15/chb15-summary.txt).

In [ ]:
"""Revision 2.1: explicit montage selection; compatible with version 2 checkpoints.

Execute AFTER the pipeline definitions, BEFORE Section 7, in the same namespace.
Does not alter inventory, folds, eligibility, scalers, or already completed files.
"""
import copy

_duplicate_base = getattr(process_file, '_duplicate_base', process_file)

def process_file(root, row, events, cfg, dest, recording_intervals=None,
                 _base=_duplicate_base):
    selected=[]
    working=row
    if row['case'] in ('chb12','chb15'):
        labels=[canonical(x) for x in row['labels']]
        for m in row['mapping']:
            if m['name']!='T8-P8' or len(m['indices'])<2:
                continue
            indices=m['indices']
            chain=[i for i in indices if 0<i<len(labels)-1
                   and labels[i-1]=='F8-T8' and labels[i]=='T8-P8'
                   and labels[i+1]=='P8-O2']
            if len(indices)!=2 or len(chain)!=1 or m['sign']!=1:
                raise ValueError('Unrecognised T8-P8 montage: retain duplicate-channel review')
            keep=chain[0]; alternate=next(i for i in indices if i!=keep)
            if keep!=indices[0] or alternate==0 or labels[alternate-1]!='FT10-T8' or labels[alternate]!='T8-P8':
                raise ValueError('T8-P8 layout differs from the resume-compatible montage policy')
            working=copy.deepcopy(row)
            for entry in working['mapping']:
                if entry['name']=='T8-P8': entry['indices']=[keep]
            selected.append(dict(channel='T8-P8',selected_index_zero_based=keep,
                omitted_index_zero_based=alternate,
                rule='Select the longitudinal-chain entry between F8-T8 and P8-O2; omit supplemental entry following FT10-T8',
                equivalence_assumed=False))
    summary,stats=_base(root,working,events,cfg,dest,recording_intervals)
    if selected:
        # The original input row stays unchanged, keeping resume locks valid.
        # The original processor ALWAYS used indices[0] after checking equality.
        # This amendment changes acceptance of alternate signals, not the selected
        # input samples, so previously successful checkpoints remain compatible.
        with h5py.File(dest,'a') as h:
            h.attrs['montage_policy_revision']='2.1'
            h.attrs['channel_selection_json']=json.dumps(selected,sort_keys=True)
        summary['channel_selection_json']=json.dumps(selected,sort_keys=True)
        summary['montage_policy_revision']='2.1'
        run_dir=Path(dest).parents[3]
        if (run_dir/'config.json').exists():
            write_json(run_dir/'duplicate_channel_policy_2_1.json',dict(
                revision='2.1',applies_to_cases=['chb12','chb15'],
                channel='T8-P8',selection='Unique entry between F8-T8 and P8-O2; must equal original first candidate',
                omitted='Supplemental entry immediately following FT10-T8',
                assumption='Distinct signals with the same label; no equality or physiological equivalence assumed',
                existing_checkpoints='Retained: original processor selected the same first candidate',
                config_compatibility='2.0',source_edfs_modified=False,
                sources=['https://physionet.org/files/chbmit/1.0.0/chb12/chb12-summary.txt',
                         'https://physionet.org/files/chbmit/1.0.0/chb15/chb15-summary.txt']))
    return summary,stats

process_file._duplicate_base = _duplicate_base
process_file._montage_policy_revision = '2.1'
print('Montage selection patch 2.1 loaded. Existing version 2 checkpoints are compatible.')


## 4 · Protocol configuration
Freeze these research settings before inspecting model test performance. A change requires a new run.

- Four-second windows; SPH = 5 minutes; SOP = 30 minutes. Positive supervised windows fall wholly within 35 to 5 minutes before a target seizure. Decisions are timestamped at window end.
- Lead targets retain the four-hour separation rule from the previous **annotated** seizure offset. Unobserved gaps do not establish seizure freedom; history coverage remains reported.
- Required preceding context has at least 95% coverage and no gap exceeding 60 seconds. Context includes the preictal interval, fusion history and raw window. These remain provisional research rules.
- Interictal training windows remain at least four hours before and after annotated seizures. Ambiguous windows stay in the stored stream with label -1.
- `chb21` is excluded by default as a repeat session of `chb01`. If included, BOTH cases share subject ID `chb01` and always receive the same fold role.
- All subjects with usable montage recordings receive an outer test fold. A subject without eligible targets contributes alarm/coverage information; its seizure sensitivity is **undefined**, not zero.
- Each outer fold has three inner folds among development subjects. These groups balance eligible event counts using a fixed seed and never use the held-out subject's counts for inner allocation.
- Inner training must have at least two eligible targets in total and inner validation at least one. These are pooled feasibility minima, not requirements on each patient.
- The old 2/1/1 event minima and 60/20/20 time fractions remain only for the optional `propose_splits` helper for the secondary chronological analysis. They do not control the primary pipeline.

In [ ]:
import copy
CONFIG = copy.deepcopy(DEFAULTS)
# Change settings here BEFORE starting a new audit.
RESUME_RUN = '/content/gdrive/MyDrive/mvPIKN_preprocessing/run_20261028T165011846018Z'
# Configurations from version 2.0 are compatible with this documented amendment.
print(json.dumps(CONFIG, indent=2))

## 5 · Audit files and reconcile the documented chb07 exception
The global `RECORDS-WITH-SEIZURES` index lists chb07_18 but omits chb07_19. The patient summary lists zero seizures in chb07_18 and a seizure at 13,688–13,831 seconds in chb07_19. PhysioNet provides `chb07_19.edf.seizures`.

The loader accepts this **specific** index discrepancy only when:
1. The index still has that exact discrepancy.
2. The summary has the documented event values.
3. The binary chb07_19 annotation confirms the onset/offset at 256 Hz.
4. There is no unexpected chb07_18 annotation file.

The resolution and annotation SHA256 are saved in `annotation_corrections.json`. Raw files and the source index are unchanged. Other disagreements still block processing. Ensure the AWS sync downloaded `.seizures` files as well as EDFs and summaries.

EDF dates define the timeline. Summary clocks are checked when present. Header availability is not sample-level QC; duplicate-channel samples are checked during processing.

In [ ]:
if RESUME_RUN is None:
    RUN, inventory, events, issues = audit_dataset(RAW_ROOT, OUTPUT_ROOT, CONFIG)
else:
    RUN = Path(RESUME_RUN)
    CONFIG = json.loads((RUN/'config.json').read_text())
    if CONFIG.get('pipeline_version') != '2.0':
        raise RuntimeError('Version 1 runs cannot be resumed here. Set RESUME_RUN=None.')
    inv_file = RUN/'locked_inventory.json' if (RUN/'locked_inventory.json').exists() else RUN/'inventory.json'
    event_file = RUN/'locked_events.json' if (RUN/'locked_events.json').exists() else RUN/'events.json'
    inventory = json.loads(inv_file.read_text())
    events = json.loads(event_file.read_text())
    issues = pd.read_csv(RUN/'audit_issues.csv').to_dict('records')
    print('Resuming saved configuration and inventory:', RUN)

print('Run directory:', RUN)
display(pd.read_csv(RUN/'cohort_audit.csv'))
display(pd.read_csv(RUN/'audit_issues.csv'))
print('Blocking issues:', sum(bool(p['blocking']) for p in issues))
print('Documented annotation corrections:')
display(pd.DataFrame(json.loads((RUN/'annotation_corrections.json').read_text())))

## 6 · Create nested subject folds
Each outer fold holds out one **entire subject**. Every other subject belongs to the development set. Inner training/validation subject groups are used for model selection, early-stopping choices, fusion settings, and thresholds. The outer test subject is never available for those choices.

For final outer evaluation, refit the selected model on all development subjects using the training duration chosen internally. Select a single threshold using inner out-of-fold development predictions; do not optimise it on the held-out subject. Apply the final model, outer-refit scaler, and fixed alarm policy to that subject's chronological stream.

`fold_manifest.json` is the authoritative assignment. The `pool` field in inventory/events means stored once, not a train/test role. `target_eligibility.csv` retains every annotated event with its eligibility result. `fold_subject_membership.csv` makes all roles inspectable. Fold IDs and assignments must remain fixed during processing.

Patients with no eligible targets remain in the fold table, labelled `alarm_and_coverage_only`. Their seizure sensitivity is not calculated.

In [ ]:
if any(bool(p['blocking']) for p in issues):
    raise RuntimeError('Resolve blocking audit issues before constructing folds.')
FOLDS = prepare_subject_folds(inventory, events, CONFIG)
write_json(RUN/'fold_manifest.json', FOLDS)
write_json(RUN/'proposed_inventory.json', inventory)
write_json(RUN/'proposed_events.json', events)
pd.DataFrame(events).to_csv(RUN/'target_eligibility.csv', index=False)
fold_report = fold_table(FOLDS)
fold_report.to_csv(RUN/'fold_report.csv', index=False)
membership=[]
for fold in FOLDS:
    for subject in fold['development_subjects']:
        membership.append(dict(fold=fold['id'],scope='outer_refit',subject=subject,role='train'))
    for subject in fold['test_subjects']:
        membership.append(dict(fold=fold['id'],scope='outer_refit',subject=subject,role='test'))
    for inner in fold['inner_folds']:
        for role,key in [('train','train_subjects'),('validation','validation_subjects')]:
            for subject in inner[key]:
                membership.append(dict(fold=fold['id'],scope=inner['id'],subject=subject,role=role))
pd.DataFrame(membership).to_csv(RUN/'fold_subject_membership.csv',index=False)
display(fold_report)
print('Eligible targets across outer tests:',fold_report.test_eligible_targets.sum())
print('Subjects with forecast targets:',(fold_report.test_eligible_targets>0).sum())
print('Review these reports before enabling preprocessing.')

## 7 · Process once and build fold-specific scalers
Set `RUN_PREPROCESSING=True` after reviewing the protocol and audit. Data are saved once per EDF under `processed/<case>/pool/`; folds reference these shared files. No oversampling, augmentation, or model training occurs here.

Filtering and spectral extraction use fixed operations. Per-case sufficient statistics are saved as intermediate summaries. Each **inner scaler aggregates only its inner training subjects**; each **outer-refit scaler aggregates only that outer fold's development subjects**. The held-out subject never contributes to either. The saved `fit_provenance.json` identifies the exact contributors. Scaling is sample-weighted across pooled valid labelled training windows.

Completed EDFs have checkpoints. For a Colab disconnect, remount Drive, set `RESUME_RUN` to this version 2 run, and rerun Sections 4–7. Changed data, settings, or fold assignments require a new run.

Review `status.json` after completion. No-input target events remain reported; they are not silently removed from the evaluation denominator. Cohort and alarm-scoring decisions still need to account explicitly for missing signal and unusable forecast intervals.

Revision 2.1 retries the failed duplicate-channel recordings using the explicit longitudinal-chain selection. Existing successful checkpoints are retained. New files record selected and omitted channel indices in HDF5 attributes and processing summaries. The run-level policy is saved in `duplicate_channel_policy_2_1.json`.

In [ ]:
RUN_PREPROCESSING = False
if RUN_PREPROCESSING:
    processing_report = preprocess(RAW_ROOT, RUN, inventory, events, issues, CONFIG, folds=FOLDS)
    display(processing_report.groupby('case')[
        ['windows','valid_windows','valid_hours','label_0_valid','label_1_valid','label_2_valid']].sum())
    display(pd.read_csv(RUN/'event_input_coverage.csv'))
    print(json.dumps(json.loads((RUN/'status.json').read_text()),indent=2))
else:
    print('Audit and fold manifests are ready. Enable processing after reviewing them.')

## 7B · Audit forecast targets separately from training labels (revision 2.2)
Run this cell on the existing completed run; Sections 3–7 are not required for this standalone audit. Google Drive must be mounted.

The four-hour postictal rule can exclude training labels in an otherwise annotation-eligible target’s preictal period. `valid_signal_windows` counts valid EEG regardless of training label; `valid_training_windows` additionally requires label 1 and the matching event ID. Flag counts can overlap. Fractions use the full configured SOP, so gaps and excluded windows remain visible.

All eligible targets remain in the event-evaluation cohort, including targets with no supervised labels. Replay and scoring must not filter the timeline to label 0/1 windows. Availability of valid EEG alone does not guarantee enough fusion history or full forecast coverage.

Outputs: `event_training_label_audit_v2_2.csv` and `training_label_audit_status_v2_2.json`. The original `status.json` and `event_input_coverage.csv` remain historical processing reports; their zero-input warning refers to **supervised preictal input**, not necessarily invalid EEG. This audit does not approve training or finalize scoring rules.


In [ ]:
# Revision 2.2: standalone audit; no EDF processing or label/policy changes.
from pathlib import Path
import json
import numpy as np
import pandas as pd
import h5py

def audit_training_label_coverage(run):
    run = Path(run)
    cfg = json.loads((run / 'config.json').read_text())
    events = json.loads((run / 'locked_events.json').read_text())
    inventory = json.loads((run / 'locked_inventory.json').read_text())
    original_status = json.loads((run / 'status.json').read_text())
    if original_status.get('status') == 'incomplete':
        raise RuntimeError('Finish failed recordings before running this audit.')
    selected = [r for r in inventory if r['signal_ok'] and r['split'] in ['pool','train','validation','test']]
    files = {}
    for r in selected:
        p = run / 'processed' / r['case'] / r['split'] / (Path(r['relative_path']).stem + '.h5')
        if not p.exists():
            raise FileNotFoundError(p)
        files.setdefault((r['case'], r['split']), []).append(p)
    report = []
    for e in events:
        if not e['eligible_target'] or e['split'] == 'unassigned':
            continue
        a = e['onset_s'] - cfg['sph_s'] - cfg['sop_s']
        b = e['onset_s'] - cfg['sph_s']
        counts = dict(period_windows=0, valid_signal_windows=0,
                      training_label_windows=0, valid_training_windows=0,
                      postictal_excluded_windows=0, other_label_excluded_windows=0,
                      flatline_windows=0, warmup_windows=0, nonfinite_windows=0)
        for p in files.get((e['case'], e['split']), []):
            with h5py.File(p, 'r') as h:
                starts = h['window_start_s'][:]
                ends = h['decision_s'][:]
                mask = (starts >= a) & (ends <= b)
                if not mask.any():
                    continue
                s, t = starts[mask], ends[mask]
                valid = h['valid_signal'][:][mask]
                label = (h['label'][:][mask] == 1) & (h['target_event_id'].asstr()[:][mask] == e['event_id'])
                post = np.zeros(len(s), dtype=bool)
                for prev in events:
                    if prev['case'] == e['case']:
                        post |= (s < prev['offset_s'] + cfg['postictal_s']) & (t > prev['offset_s'])
                counts['period_windows'] += len(s)
                counts['valid_signal_windows'] += int(valid.sum())
                counts['training_label_windows'] += int(label.sum())
                counts['valid_training_windows'] += int((valid & label).sum())
                counts['postictal_excluded_windows'] += int((~label & post).sum())
                counts['other_label_excluded_windows'] += int((~label & ~post).sum())
                for name in ['flatline', 'warmup', 'nonfinite']:
                    counts[name + '_windows'] += int(h[name + '_flag'][:][mask].sum())
        n, labelled, usable = (counts[k] for k in ['period_windows','training_label_windows','valid_training_windows'])
        category = ('no_recorded_windows' if n == 0 else
                    'no_training_labels' if labelled == 0 else
                    'all_training_labels_signal_invalid' if usable == 0 else
                    'partial_training_labels' if labelled < n else
                    'labels_available_for_all_recorded_windows')
        report.append(dict(case=e['case'], event_id=e['event_id'], split=e['split'],
                           forecast_target_eligible=True, retain_in_event_evaluation=True,
                           training_input_category=category, **counts,
                           invalid_training_windows=labelled-usable,
                           valid_training_minutes=usable*cfg['window_s']/60,
                           valid_training_fraction_of_SOP=usable*cfg['window_s']/cfg['sop_s'],
                           valid_signal_fraction_of_SOP=counts['valid_signal_windows']*cfg['window_s']/cfg['sop_s']))
    result = pd.DataFrame(report)
    if result.empty:
        raise RuntimeError('No eligible targets found in locked events.')
    result.to_csv(run / 'event_training_label_audit_v2_2.csv', index=False)
    note = dict(revision='2.2', status='audit_complete_evaluation_policy_review_required',
                eligible_forecast_targets=len(result),
                targets_with_valid_training_input=int((result.valid_training_windows > 0).sum()),
                targets_without_training_labels=int((result.training_label_windows == 0).sum()),
                source_status=original_status,
                note='Training-label availability does not redefine forecast eligibility. Retain all locked eligible targets. Signal availability is not complete model prediction coverage: fusion history, gaps and alarm scoring still require explicit handling. No HDF5, labels, folds, scalers, eligibility, or original status changed.')
    (run / 'training_label_audit_status_v2_2.json').write_text(json.dumps(note, indent=2))
    return result, note

AUDIT_RUN = Path('/content/gdrive/MyDrive/mvPIKN_preprocessing/run_20261028T165011846018Z')
label_audit, label_audit_status = audit_training_label_coverage(AUDIT_RUN)
display(label_audit)
print(json.dumps(label_audit_status, indent=2))


## 8 · Inspect a completed output
Each HDF5 file contains:

| Dataset | Meaning |
|---|---|
| `raw_uv` | Filtered EEG, `[windows, 18, 1024]`, float32 µV |
| `features` | Spectral view, `[windows, 18, 5]` |
| `label` | -1 uncertain/excluded, 0 interictal, 1 preictal, 2 ictal |
| `window_start_s`, `decision_s` | Elapsed seconds from the case origin; gaps remain real gaps |
| `valid_signal` and quality flags | Signal-quality and filter transient masks |
| `target_event_id` | Associated eligible preictal target, otherwise empty |
| `pair_with_previous` | Both this and the previous window have valid signals in the same EDF |
| `supervised_pair_with_previous` | Both windows also have nonnegative labels |
| `future_annotation_coverage`, `future_annotation_max_gap_s` | Coverage of the future SPH/SOP target interval inside this partition |

For a Koopman pair, index `i` with `pair_with_previous[i] = True` refers to **(i−1, i)**, separated by four seconds. The first window is always unpaired. Training may use only training files; this pairing mask does not grant access to validation/test data.

Scalers are specific to the outer fold and fit scope. Inner validation uses that inner fold's scaler; outer testing uses the outer-refit scaler. Do not reuse a scaler across outer folds. Saved arrays remain in physical units, and the loader normalises lazily.

`fold_files` supplies explicit file lists. For inner fitting use `scope='inner_01', role='train'` or `'validation'`; for the final model use `scope='outer_refit', role='train'` or `'test'`. Model training must select valid labelled windows only from the appropriate training list. Validation and test alarm replay retain chronology, quality masks, and true recording gaps.

In [ ]:
fold = FOLDS[0]
print('Example fold:',fold['id'])
train_files = fold_files(RUN, inventory, fold, scope='inner_01',role='train')
validation_files = fold_files(RUN, inventory, fold, scope='inner_01',role='validation')
test_files = fold_files(RUN, inventory, fold, scope='outer_refit',role='test')
print('EDF counts:',len(train_files),'inner train;',len(validation_files),'inner validation;',len(test_files),'outer test')
scaler = RUN/'folds'/fold['id']/'inner_01'/'train_scaler.npz'
if train_files and Path(train_files[0]).exists() and scaler.exists():
    with h5py.File(train_files[0],'r') as h:
        indices=np.flatnonzero(h['valid_signal'][:]& (h['label'][:]>=0))
    if len(indices):
        raw,spectral,label=load_window(train_files[0],int(indices[0]),scaler)
        print('Example normalised training input:',raw.shape,spectral.shape,'label:',label)
else:
    print('Run preprocessing to materialise these files and scalers.')

## Evaluation handoff and limitations
- Inspect `status.json`, `processing_errors.csv`, event-input coverage, and per-patient class counts before training. Proposed eligibility is not a guarantee of usable signal.
- Preserve the four-hour target and postictal rules for the primary run. A sensitivity analysis that changes them must be versioned and reported, not selected because it yields better test results.
- The full chronological stream stays on disk. Label -1 excludes supervised training, not automatically alarm replay. Define handling of ictal/postictal intervals, gaps, poor-signal abstention, incomplete horizons, and the false-alarm denominator before final evaluation.
- Each subject receives outer predictions once. Report event counts, eligible monitoring hours, event sensitivity where defined, false alarms/hour, warning burden, and uncertainty. Zero-target subjects remain visible and do not receive an invented sensitivity of zero.
- The old chronological `propose_splits` helper remains available as a secondary analysis only. Run it on copies of inventory/events and store its outputs separately; it must not mutate the primary pool or fold manifest.

### Sources
- [CHB-MIT documentation](https://physionet.org/content/chbmit/1.0.0/).
- [Global seizure-file index](https://physionet.org/files/chbmit/1.0.0/RECORDS-WITH-SEIZURES), [chb07 summary](https://physionet.org/files/chbmit/1.0.0/chb07/chb07-summary.txt), and [chb07 annotation listing](https://physionet.org/content/chbmit/1.0.0/chb07/).
- [WFDB annotation reader](https://wfdb.readthedocs.io/en/latest/io.html#wfdb.io.rdann).
- [SciPy forward filtering](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.sosfilt.html) and [Welch PSD](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.welch.html).
- [Winterhalder et al. (2003)](https://doi.org/10.1016/S1525-5050(03)00105-7), [Snyder et al. (2008)](https://pmc.ncbi.nlm.nih.gov/articles/PMC2888045/), [Andrzejak et al. (2009)](https://doi.org/10.1016/j.clinph.2009.05.019).

**Validation:** synthetic EDF pipeline and resume checks, channel/label checks, disjoint subject folds, repeat-session grouping, annotation exception verification, and isolation of held-out/validation statistics from training scalers. Full Drive-data execution remains to be performed in Colab.

Revision 2.1 validation additionally checks exact equality of processed outputs and training sufficient statistics against explicit first-channel selection, unchanged inventory, repeatable installation, and rejection of unrelated duplicates. The two T8-P8 traces are not assumed physiologically equivalent; the fixed montage location is the selection rule, and signal-quality flags still apply to the retained trace.